# T+1 자전거 대여량 예측 - 학생용

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
raw = pd.read_csv("SeoulBikeData.csv", encoding="latin1")
bike = raw.copy()
bike.columns = [
    "date", "rented_count", "hour", "temperature", "humidity",
    "wind_speed", "visibility", "dew_point", "solar_radiation",
    "rainfall", "snowfall", "season", "holiday", "functioning_day"
]
bike["date"] = pd.to_datetime(bike["date"], format="%d/%m/%Y")
display(bike.head())

,date,rented_count,hour,temperature,humidity,wind_speed,visibility,dew_point,solar_radiation,rainfall,snowfall,season,holiday,functioning_day
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [3]:
bike["datetime"] = bike["date"] + pd.to_timedelta(bike["hour"], unit="h")
bike = bike.sort_values("datetime").copy()
bike["target_datetime"] = bike["datetime"].shift(-1)
bike["next_rented_count"] = bike["rented_count"].shift(-1)
bike["next_functioning_day"] = bike["functioning_day"].shift(-1)
display(bike[["datetime", "target_datetime", "rented_count", "next_rented_count"]].head())

,datetime,target_datetime,rented_count,next_rented_count
0,2017-12-01 00:00:00,2017-12-01 01:00:00,254,204.0
1,2017-12-01 01:00:00,2017-12-01 02:00:00,204,173.0
2,2017-12-01 02:00:00,2017-12-01 03:00:00,173,107.0
3,2017-12-01 03:00:00,2017-12-01 04:00:00,107,78.0
4,2017-12-01 04:00:00,2017-12-01 05:00:00,78,100.0


In [4]:
one_hour_later = bike["target_datetime"] - bike["datetime"] == pd.Timedelta(hours=1)
operating = bike[
    (bike["functioning_day"] == "Yes")
    & (bike["next_functioning_day"] == "Yes")
    & one_hour_later
].copy()
train = operating[operating["target_datetime"] < "2018-09-01"].copy()
test = operating[operating["datetime"] >= "2018-09-01"].copy()
explore = train.copy()
print("학습 기록 수:", len(train))
print("평가 기록 수:", len(test))

학습 기록 수: 6525
평가 기록 수: 1926


In [5]:
X_test = test[["hour", "temperature", "humidity", "rainfall"]]
y_test = test["next_rented_count"]
baseline_pred = [train["next_rented_count"].median()] * len(test)
linear_four = LinearRegression()
tree = DecisionTreeRegressor(max_depth=6, random_state=0)
forest = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=0)

| 열 | 뜻 |
|---|---|
| `datetime` | 입력 시각 T |
| `target_datetime` | 예측 대상 시각 T+1 |
| `hour` | 현재 시각 T의 시간대, 0~23시 |
| `temperature` | T 시점의 기온, ℃ |
| `humidity` | T 시점의 습도, % |
| `rainfall` | T 시점의 강수량, mm |
| `rented_count` | T 시점의 실제 대여 건수 |
| `next_rented_count` | T+1 시점의 실제 대여 건수, 예측할 정답 |

## Part 1 - 데이터의 특징을 코드로 확인하기

### 1-1. T와 T+1의 짝을 출력하는 코드를 구현하라

train에서 datetime, target_datetime, rented_count, next_rented_count 네 열의 앞 5행만 표로 출력하라.

In [ ]:
# 코드 작성

### 1-2. 다음 시간 대여량의 요약 통계를 출력하는 코드를 구현하라

train의 next_rented_count 열에 대한 요약 통계를 출력하라.

In [ ]:
# 코드 작성

### 1-3. 열별 결측값 개수를 출력하는 코드를 구현하라

train의 모든 열에 대해 결측값 개수를 계산하여 출력하라.

In [ ]:
# 코드 작성

### 1-4. 기온과 다음 시간 대여량의 상관계수를 출력하는 코드를 구현하라

explore의 temperature와 next_rented_count 두 열의 상관계수 표를 출력하라.

In [ ]:
# 코드 작성

### 1-5. 입력 시간대별 다음 시간 평균 대여량을 출력하는 코드를 구현하라

explore를 hour별로 묶어 next_rented_count의 평균을 출력하라.

In [ ]:
# 코드 작성

### 1-6. 비가 오는 기록의 다음 시간 평균 대여량을 출력하는 코드를 구현하라

explore에서 rainfall이 0보다 큰 기록만 사용하여 next_rented_count의 평균을 출력하라.

In [ ]:
# 코드 작성

### 1-7. 습도와 다음 시간 대여량의 상관계수를 출력하는 코드를 구현하라

explore의 humidity와 next_rented_count 두 열의 상관계수 표를 출력하라.

In [ ]:
# 코드 작성

## Part 2 - 선형 회귀로 다음 시간 대여량 예측하기

### 2-1. 학습 입력 표를 만드는 코드를 구현하라

train에서 hour, temperature, humidity, rainfall을 이 순서로 선택한 X_train을 만들고 앞 5행을 출력하라.

In [ ]:
# 코드 작성

### 2-2. 학습 정답 열을 만드는 코드를 구현하라

train의 next_rented_count를 y_train에 저장하고 앞 5행을 출력하라.

In [ ]:
# 코드 작성

### 2-3. 선형 회귀를 학습하는 코드를 구현하라

제공된 linear_four 모델을 X_train과 y_train으로 학습하라.

In [ ]:
# 코드 작성

### 2-4. 선형 회귀의 다음 시간 예측값을 구하는 코드를 구현하라

linear_four로 X_test를 예측한 linear_four_pred를 만들고 앞 5개 예측값을 출력하라.

In [ ]:
# 코드 작성

### 2-5. 선형 회귀의 MAE를 계산하는 코드를 구현하라

y_test와 linear_four_pred의 MAE를 linear_four_mae에 저장하고 출력하라.

In [ ]:
# 코드 작성

### 2-6. 중앙값 기준 예측의 MAE를 계산하는 코드를 구현하라

y_test와 제공된 baseline_pred의 MAE를 baseline_mae에 저장하고 출력하라.

In [ ]:
# 코드 작성

## Part 3 - 나무 모델로 다음 시간 대여량 예측하기

### 3-1. 의사결정나무를 학습하는 코드를 구현하라

제공된 tree 모델을 X_train과 y_train으로 학습하라.

In [ ]:
# 코드 작성

### 3-2. 의사결정나무의 다음 시간 예측값을 구하는 코드를 구현하라

tree로 X_test를 예측한 tree_pred를 만들고 앞 5개 예측값을 출력하라.

In [ ]:
# 코드 작성

### 3-3. 의사결정나무의 MAE를 계산하는 코드를 구현하라

y_test와 tree_pred의 MAE를 tree_mae에 저장하고 출력하라.

In [ ]:
# 코드 작성

### 3-4. 랜덤 포레스트를 학습하는 코드를 구현하라

제공된 forest 모델을 X_train과 y_train으로 학습하라.

In [ ]:
# 코드 작성

### 3-5. 랜덤 포레스트의 다음 시간 예측값을 구하는 코드를 구현하라

forest로 X_test를 예측한 forest_pred를 만들고 앞 5개 예측값을 출력하라.

In [ ]:
# 코드 작성

### 3-6. 랜덤 포레스트의 MAE를 계산하는 코드를 구현하라

y_test와 forest_pred의 MAE를 forest_mae에 저장하고 출력하라.

In [ ]:
# 코드 작성

### 3-7. 네 방법의 MAE 비교표를 만드는 코드를 구현하라

중앙값 기준, 선형 회귀, 의사결정나무, 랜덤 포레스트의 MAE를 방법과 평가 MAE 두 열의 comparison 표로 만들어 출력하라.

In [ ]:
# 코드 작성